In [ ]:
import os
import yaml
import pandas as pd
import numpy as np
from plotnine import *

from tqdm import tqdm
from sklearn.metrics import r2_score


In [ ]:
config_path = '../../run_config_local.yaml'
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

traits=config.get('traits')
# traits

In [ ]:
base_dir = '/s/project/geno2pheno/funcrvp/paper_revisions'
lm_cov = f"{base_dir}/predictions/all_traits_covariates_only_phenopred_filteredv3.pq"

rvat_path = f"{base_dir}/predictions/ukbb_wes_500k_DeepRVAT_final_090924_medianshifted/rvat_sampling"
funcrvp_path = f"{base_dir}/predictions/ukbb_wes_500k_DeepRVAT_final_090924_medianshifted/pops_mat_pca256_omics/funcrvp_better_filteredv3_sampling"

model_dir_list = []
rvat_dir_list = []
lm_cov_path_list = []
for s in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    model_dir_list.append(funcrvp_path + str(s))
    rvat_dir_list.append(rvat_path + str(s))
    lm_cov_path_list.append(rvat_path + str(s) + '_onlycov')

## Save funcrvp results

In [ ]:
skip_list = []
for model_dir in tqdm(model_dir_list):
    beta_trait_list = []
    pred_trait_list = []
    for trait in tqdm(traits):
        try:
            # Load the predictions
            beta_file = os.path.join(model_dir, f"{trait}_betas.pq")
            beta_trait_list.append(pd.read_parquet(beta_file))

            pred_file = os.path.join(model_dir, f"{trait}_phenopred.pq")  
            pred_trait_list.append(pd.read_parquet(pred_file))
        
        except FileNotFoundError:
            skip_list.append(f"{trait}: {model_dir}")

    # Concatenate the dataframes
    beta_df = pd.concat(beta_trait_list, axis=0)
    pred_df = pd.concat(pred_trait_list, axis=0).reset_index()
    # Save the concatenated dataframes to parquet files
    beta_df.to_parquet(os.path.join(model_dir, "all_traits_betas.pq"))
    pred_df.to_parquet(os.path.join(model_dir, "all_traits_phenopred.pq"))

print(skip_list)

## Save RVAT results

In [ ]:
for rvat_dir in tqdm(rvat_dir_list):
    beta_trait_list = []
    for trait in tqdm(traits):
        beta_file = os.path.join(rvat_dir, f"{trait}_rvat.pq")
        beta_trait_list.append(pd.read_parquet(beta_file))

    # Concatenate and save the dataframes
    beta_df = pd.concat(beta_trait_list, axis=0)
    beta_df.to_parquet(os.path.join(rvat_dir, "all_traits_rvat.pq"))

In [ ]:
p_thresh_list = ['0.05']
skip_list = []
for rvat_dir in tqdm(rvat_dir_list):
    for p_thresh in tqdm(p_thresh_list):
        pred_trait_list = []
        for trait in tqdm(traits):
            try:
                pred_file = os.path.join(rvat_dir, f"{trait}_phenopred_{p_thresh}.pq")  
                pred_trait_list.append(pd.read_parquet(pred_file))
            except:
                skip_list.append(pred_file)
        pred_df = pd.concat(pred_trait_list, axis=0).reset_index()
        pred_df.to_parquet(os.path.join(rvat_dir, f"all_traits_phenopred_{p_thresh}.pq"))

print(skip_list)

In [ ]:
skip_list

## LM - Covariates

In [ ]:
p_thresh_list = ['0.05']
skip_list = []
for rvat_dir in tqdm(lm_cov_path_list):
    for p_thresh in tqdm(p_thresh_list):
        pred_trait_list = []
        for trait in tqdm(traits):
            try:
                pred_file = os.path.join(rvat_dir, f"{trait}_phenopred_{p_thresh}.pq")  
                pred_trait_list.append(pd.read_parquet(pred_file))
            except:
                skip_list.append(pred_file)
        pred_df = pd.concat(pred_trait_list, axis=0)
        pred_df.to_parquet(os.path.join(rvat_dir, f"all_traits_phenopred_{p_thresh}.pq"))

print(skip_list)